<a href="https://colab.research.google.com/github/elizelton/Liveness_Detection/blob/main/Liveness_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preparando ambiente

# 1. Preparando modelo

In [ ]:
# import the necessary packages
from keras.models import Sequential
#from keras.layers.normalization import BatchNormalization
from keras.layers.convolutional import MaxPooling2D
from keras.layers.convolutional import Conv2D
from keras.layers.core import Activation
from keras.layers.core import Flatten
from keras.layers.core import Dropout
from keras.layers.core import Dense
from keras.layers import GlobalAveragePooling2D
from keras.layers import AveragePooling2D
from keras.layers import ZeroPadding2D
from keras.layers import Input
from keras.layers import add
from keras.models import Model
from keras import backend as K
# from tensorflow.keras.applications import ResNet50
# Load VGG16 without the top FC Layers.
from tensorflow.keras.applications import VGG16

class LivenessNet:
    @staticmethod
    def build(width, height, depth, classes):
      print('ok')
      vgg_conv = VGG16(weights='imagenet', include_top=False, input_shape=(width, height, depth))

      vgg_conv.summary()

      # Fine Tunning VGG16
      # Create 'model' from vgg_conv adding dense layers and reducing the number of classes to 10

      model = Sequential()
      model.add(vgg_conv)
      model.add(Flatten())
      model.add(Dense(1024, activation='relu'))
      model.add(Dropout(0.5))
      model.add(Dense(classes, activation='softmax', name="fc7" ))

      # Freeze the layers except the last 4 layers
      for layer in model.layers[:-4]:
        layer.trainable = False

      # Check the trainable status of the individual layers
      for layer in model.layers:
        print(layer, layer.trainable)

      return model

ModuleNotFoundError: No module named 'keras.layers.convolutional'

# 2. Preparando imagens

# 3. Treinamento

In [ ]:
     # USAGE
# python train_liveness.py --dataset dataset --model liveness.model --le le.pickle

# definir o backend do matplotlib para que as figuras possam ser salvas no fundo
import matplotlib
matplotlib.use("Agg")

# importe os pacotes necessários
#de livenessnet import LivenessNet
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from keras.utils import np_utils
from imutils import paths
import matplotlib.pyplot as plt
import numpy as np
import argparse
import pickle
import cv2
import os
from google.colab import files

# inicializar a taxa de aprendizagem inicial, tamanho do lote e número de
# épocas para treinar
INIT_LR = 1e-4
BS = 128
EPOCHS = 50
PATHDRIVE="/content/drive/MyDrive/liveness"
WIDTH=224
HEIGHT=224

# pegue a lista de imagens em nosso diretório de conjunto de dados e inicialize
# a lista de dados (ou seja, imagens) e imagens de classe
print("[INFO] loading images...")
imagePaths = list(paths.list_images(os.path.sep.join([PATHDRIVE, "dataset"])))
data = []
labels = []
trainX = []
testX = []


for imagePath in imagePaths:
  name = imagePath.split(os.path.sep)[-1].split(".")[-2]
  if len(name.split("_")) < -4 :
    name = name.split("_")[-5]

  # extraia o rótulo da classe do nome do arquivo, carregue a imagem e
  # redimensione-o para 32x32 pixels fixos, ignorando a proporção
  label = imagePath.split(os.path.sep)[-2]
  image = cv2.imread(imagePath)
  image = cv2.resize(image, (WIDTH, HEIGHT))
  print(imagePath)
  # atualizar as listas de dados e rótulos, respectivamente

  data.append(image)
  labels.append(label)


# converter os dados em uma matriz NumPy e, em seguida, pré-processá-los escalando
# todas as intensidades de pixel no intervalo [0, 1]
data = np.array(data, dtype="float") / 255.0
'DATA'
data
'=========================='
# codificar os rótulos (que atualmente são strings) como inteiros e então
# codifique-os de uma vez
le = LabelEncoder()
labels = le.fit_transform(labels)
labels = np_utils.to_categorical(labels, 2)

# particionar os dados em divisões de treinamento e teste usando 75% de
# os dados para treinamento e os 25% restantes para teste
(trainX, testX, trainY, testY) = train_test_split(data, labels, test_size=0.25, random_state=42)

print(trainX)
print("----------------")
print(trainY)
print("----------------")
print(testX)
print("----------------")
print(testY)
# construir o gerador de imagem de treinamento para aumento de dados
aug = ImageDataGenerator(rotation_range=20, zoom_range=0.15,
  width_shift_range=0.2, height_shift_range=0.2, shear_range=0.15,
  horizontal_flip=True, fill_mode="nearest")

# inicializar o otimizador e o modelo
print("[INFO] compiling model...")
opt = Adam(learning_rate=INIT_LR, decay=INIT_LR / EPOCHS)
model = LivenessNet.build(width=WIDTH, height=HEIGHT, depth=3,
  classes=len(le.classes_))
model.compile(loss="binary_crossentropy", optimizer=opt,
  metrics=["accuracy"])
model.summary()

# treinar a rede
print("[INFO] training network for {} epochs...".format(EPOCHS))
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1)
H = model.fit(aug.flow(trainX, trainY, batch_size=BS),
  validation_split=0.2, steps_per_epoch=len(trainX) // BS,
  epochs=EPOCHS)

# avalie a rede
print("[INFO] evaluating network...")
predictions = model.predict(testX, batch_size=BS)
print(classification_report(testY.argmax(axis=1),
  predictions.argmax(axis=1), target_names=le.classes_))

# salve a rede no disco
print("[INFO] serializing network to '{}'...".format("liveness.model"))
print(model)
model.save("liveness.model")

# salve o codificador de etiqueta no disco
f = open("le.pickle", "wb")
f.write(pickle.dumps(le))
f.close()

# traçar a perda e a precisão do treinamento
plt.style.use("ggplot")
plt.figure()
print(H.history)
plt.plot(np.arange(0, EPOCHS), H.history["loss"], label="train_loss")
plt.plot(np.arange(0, EPOCHS), H.history["val_loss"], label="val_loss")
plt.plot(np.arange(0, EPOCHS), H.history["accuracy"], label="train_acc")
plt.plot(np.arange(0, EPOCHS), H.history["val_accuracy"], label="val_acc")
plt.title("Training Loss and Accuracy on Dataset")
plt.xlabel("Epoch #")
plt.ylabel("Loss/Accuracy")
plt.legend(loc="lower left")
plt.savefig("plot.png")

[INFO] loading images...


ValueError: ignored

In [ ]:
#@title Example form fields
#@markdown Forms support many types of fields.

no_type_checking = ''  #@param
string_type = 'example'  #@param {type: "string"}
slider_value = 142  #@param {type: "slider", min: 100, max: 200}
number = 102  #@param {type: "number"}
date = '2010-11-05'  #@param {type: "date"}
pick_me = "monday"  #@param ['monday', 'tuesday', 'wednesday', 'thursday']
select_or_input = "apples" #@param ["apples", "bananas", "oranges"] {allow-input: true}
#@markdown ---


# 4. Importações camera e canva

In [ ]:
# import dependencies
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import cv2
import numpy as np
import PIL
import io
import html
import time

In [ ]:
# function to convert the JavaScript object into an OpenCV image
def js_to_image(js_reply):
  """
  Params:
          js_reply: JavaScript object containing image from webcam
  Returns:
          img: OpenCV BGR image
  """
  # decode base64 image
  image_bytes = b64decode(js_reply.split(',')[1])
  # convert bytes to numpy array
  jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
  # decode numpy array into OpenCV BGR image
  img = cv2.imdecode(jpg_as_np, flags=1)

  return img

# function to convert OpenCV Rectangle bounding box image into base64 byte string to be overlayed on video stream
def bbox_to_bytes(bbox_array):
  """
  Params:
          bbox_array: Numpy array (pixels) containing rectangle to overlay on video stream.
  Returns:
        bytes: Base64 image byte string
  """
  # convert array into PIL image
  bbox_PIL = PIL.Image.fromarray(bbox_array, 'RGBA')
  iobuf = io.BytesIO()
  # format bbox into png for return
  bbox_PIL.save(iobuf, format='png')
  # format return string
  bbox_bytes = 'data:image/png;base64,{}'.format((str(b64encode(iobuf.getvalue()), 'utf-8')))

  return bbox_bytes

In [ ]:
# JavaScript to properly create our live video stream using our webcam as input
def video_stream():
  js = Javascript('''
    var video;
    var div = null;
    var stream;
    var captureCanvas;
    var imgElement;
    var labelElement;

    var pendingResolve = null;
    var shutdown = false;

    function removeDom() {
       stream.getVideoTracks()[0].stop();
       video.remove();
       div.remove();
       video = null;
       div = null;
       stream = null;
       imgElement = null;
       captureCanvas = null;
       labelElement = null;
    }

    function onAnimationFrame() {
      if (!shutdown) {
        window.requestAnimationFrame(onAnimationFrame);
      }
      if (pendingResolve) {
        var result = "";
        if (!shutdown) {
          captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
          result = captureCanvas.toDataURL('image/jpeg', 0.8)
        }
        var lp = pendingResolve;
        pendingResolve = null;
        lp(result);
      }
    }

    async function createDom() {
      if (div !== null) {
        return stream;
      }

      div = document.createElement('div');
      div.style.border = '2px solid black';
      div.style.padding = '3px';
      div.style.width = '100%';
      div.style.maxWidth = '600px';
      document.body.appendChild(div);

      const modelOut = document.createElement('div');
      modelOut.innerHTML = "<span>Status:</span>";
      labelElement = document.createElement('span');
      labelElement.innerText = 'No data';
      labelElement.style.fontWeight = 'bold';
      modelOut.appendChild(labelElement);
      div.appendChild(modelOut);

      video = document.createElement('video');
      video.style.display = 'block';
      video.width = div.clientWidth - 6;
      video.setAttribute('playsinline', '');
      video.onclick = () => { shutdown = true; };
      stream = await navigator.mediaDevices.getUserMedia(
          {video: { facingMode: "environment"}});
      div.appendChild(video);

      imgElement = document.createElement('img');
      imgElement.style.position = 'absolute';
      imgElement.style.zIndex = 1;
      imgElement.onclick = () => { shutdown = true; };
      div.appendChild(imgElement);

      const instruction = document.createElement('div');
      instruction.innerHTML =
          '<span style="color: red; font-weight: bold;">' +
          'When finished, click here or on the video to stop this demo</span>';
      div.appendChild(instruction);
      instruction.onclick = () => { shutdown = true; };

      video.srcObject = stream;
      await video.play();

      captureCanvas = document.createElement('canvas');
      captureCanvas.width = 640; //video.videoWidth;
      captureCanvas.height = 480; //video.videoHeight;
      window.requestAnimationFrame(onAnimationFrame);

      return stream;
    }
    async function stream_frame(label, imgData) {
      if (shutdown) {
        removeDom();
        shutdown = false;
        return '';
      }

      var preCreate = Date.now();
      stream = await createDom();

      var preShow = Date.now();
      if (label != "") {
        labelElement.innerHTML = label;
      }

      if (imgData != "") {
        var videoRect = video.getClientRects()[0];
        imgElement.style.top = videoRect.top + "px";
        imgElement.style.left = videoRect.left + "px";
        imgElement.style.width = videoRect.width + "px";
        imgElement.style.height = videoRect.height + "px";
        imgElement.src = imgData;
      }

      var preCapture = Date.now();
      var result = await new Promise(function(resolve, reject) {
        pendingResolve = resolve;
      });
      shutdown = false;

      return {'create': preShow - preCreate,
              'show': preCapture - preShow,
              'capture': Date.now() - preCapture,
              'img': result};
    }
    ''')

  display(js)

def video_frame(label, bbox):
  data = eval_js('stream_frame("{}", "{}")'.format(label, bbox))
  return data

# 5. Algoritmo detecção

In [ ]:
# importe os pacotes necessários
from imutils.video import VideoStream
from keras.preprocessing.image import img_to_array
from keras.models import load_model
from keras.models import model_from_json
from google.colab.patches import cv2_imshow
import numpy as np
import argparse
import imutils
import pickle
import time
import cv2
import os
WIDTH=224
HEIGHT=224

# carregar nosso detector facial serializado do disco
print("[INFO] loading face detector...")
protoPath = os.path.sep.join([PATHDRIVE, "face_detector", "deploy.prototxt"])
modelPath = os.path.sep.join([PATHDRIVE, "face_detector", "res10_300x300_ssd_iter_140000.caffemodel"])
net = cv2.dnn.readNetFromCaffe(protoPath, modelPath)

# carregue o modelo do detector de vivacidade e o codificador de etiqueta do disco
print("[INFO] loading liveness detector...")
model = load_model("liveness.model")
le = pickle.loads(open("le.pickle", "rb").read())

# inicializar o fluxo de vídeo e permitir que o sensor da câmera aqueça
print("[INFO] starting video stream...")
video_stream()
time.sleep(2.0)
label_html = 'Capturando...'
bbox = ''
# fazer um loop sobre os frames do stream de vídeo
while True:
	# pegue o quadro do stream de vídeo encadeado e redimensione-o
	# ter uma largura máxima de 600 pixels
  vs = video_frame(label_html, bbox)
  if not vs:
      break

  # converter resposta JS para imagem OpenCV
  frame = js_to_image(vs["img"])
  frame = imutils.resize(frame, width=600)

  # pegue as dimensões do quadro e converta-o em um blob
  (h, w) = frame.shape[:2]
  blob = cv2.dnn.blobFromImage(cv2.resize(frame, (300, 300)), 1.0,
		(300, 300), (104.0, 177.0, 123.0))

	# passar o blob pela rede e obter as detecções e
	# predições
  net.setInput(blob)
  detections = net.forward()

	# fazer um loop sobre as detecções
  for i in range(0, detections.shape[2]):
		# extrair a confiança (ou seja, probabilidade) associada a
		# predição
    confidence = detections[0, 0, i, 2]

		# filtrar detecções fracas
    if confidence > 0.5:
			# calcular as coordenadas (x, y) da caixa delimitadora para
			# o rosto e extrair o ROI do rosto
      box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
      (startX, startY, endX, endY) = box.astype("int")
      # garantir que a caixa delimitadora detectada fique fora do
			# dimensões da moldura
      startX = max(0, startX)
      startY = max(0, startY)
      endX = min(w, endX)
      endY = min(h, endY)
      # extrair o ROI de face e, em seguida, pré-processá-lo no exato
      # da mesma maneira que nossos dados de treinamento
      face = frame[startY:endY, startX:endX]
      face = cv2.resize(face, (WIDTH, HEIGHT))
      face = face.astype("float") / 255.0
      face = img_to_array(face)
      face = np.expand_dims(face, axis=0)

      # passar o ROI facial por meio do detector de vivacidade treinado
      # modelo para determinar se o rosto é "real" ou "falso"
      preds = model.predict(face)[0]
      j = np.argmax(preds)
      label = le.classes_[j]

      # criar sobreposição transparente para a caixa delimitadora
      bbox_array = np.zeros([480,640,4], dtype=np.uint8)

      # desenhe o rótulo e a caixa delimitadora na moldura
      label = "{}: {:.4f}".format(label, preds[j])
      bbox_array = cv2.putText(bbox_array, label, (startX, startY - 10),
				cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

      # obter caixa delimitadora de rosto para sobreposição
      bbox_array = cv2.rectangle(bbox_array, (startX, startY), (endX, endY),
      (0, 0, 255), 2)

      bbox_array[:,:,3] = (bbox_array.max(axis = 2) > 0 ).astype(int) * 255
      # converter a sobreposição de bbox em bytes
      bbox_bytes = bbox_to_bytes(bbox_array)

      # atualize o bbox para que o próximo quadro receba uma nova sobreposição
      bbox = bbox_bytes

  # mostre o quadro de saída e espere por um pressionamento de tecla
  # cv2_imshow(frame)
  key = cv2.waitKey(1) & 0xFF
  # se a tecla `q` foi pressionada, interrompa o loop
  if key == ord("q"):
      break

# faça um pouco de limpeza
cv2.destroyAllWindows()